# UDACITY Designing Your First Workflow - Tying it All Together

AWS is comprised of many services, and one of the main skills you'll develop as an ML Engineer working in AWS is in chaining these services together to accomplish specific data engineering goals. With Lambda, you've learned how to launch serverless jobs, and with Step Functions, you've learned how to create a workflow that chains jobs together. Now, you'll learn how to launch a Step Function using a Lambda job. 

Before starting this, it's important to highlight that this is not the only way to accomplish something like this. Multiple services integrate with Step Functions, and so it follows that there are multiple ways to launch Step Functions. These services, among others, include API Gateway, EventBridge, and even other Step Functions. 

Your task is to create a new lambda function that will launch the state machine you created in the **last exercise**. You'll then launch this lambda function from the command line. To find the definition of the step function you've made, click into the step function and look for the definition under the 'Definition' tab. 

First, create a new Lambda role. Attach to this role the StepFunctionsFullAccess policy. Then create a new lambda function under the default template, and attach this new role to it. Use the starter code below to help you modify the lambda handler to accomplish your task. 

As Step Function cannot execute more than once with the same name, you must update the definition with a new name. You can find the existing definition of a Step Function in the AWS Console under 'Step Functions'. In the lambda function code below, update the 'definition' with the step function definition from your last exercise, with the only difference being the, step fucntion name, processing-job name and the training-job name. 

## Exercise: Create the Lambda Function

In [ ]:
# import json
# import boto3
# import time

# client = boto3.client('stepfunctions')

# # todo, copy the definition from the last exercise and paste it below. 
# # Also change the names of step function, training job and processing job. 
# definition = """

# """

# def lambda_handler(event, context):
#     #todo 
#     client.update_state_machine(definition=definition, stateMachineArn='CHANGE THIS') 
#     # Give AWS time to register the defintion
#     time.sleep(5)
#     #todo
#     client.start_execution(input='{}', name='CHANGE THIS', stateMachineArn='CHANGE THIS') 
    
#     return {
#         'statusCode': 200,
#         'body': 'The step function has successfully launched!'
#     }


## Method 1 : Create lambda function from scratch & make sure it has required role to run the pipeline


In [13]:
code = '''
import json
import boto3

client = boto3.client('sagemaker')

def lambda_handler(event, context):
    response = client.start_pipeline_execution(
        PipelineName="l3e3-pipeline"
    )
    return {
        'statusCode': 200,
        'body': json.dumps({
            'message': 'Pipeline execution started',
            'executionArn': response['PipelineExecutionArn'],
        })
    }
'''

with open('lambda_function.py', 'w') as f:
    f.write(code)

In [15]:
# compress the lambda function into a zip file
import zipfile
with zipfile.ZipFile('lambda.zip', 'w') as z:
    z.write('lambda_function.py')

In [16]:
# 1. Create the role with a Lambda trust policy
import boto3, json

iam = boto3.client('iam')

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

role = iam.create_role(
    RoleName='lambda-launch-pipeline',
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Description='Lets Lambda start the l3e3 SageMaker pipeline',
)
role_arn = role['Role']['Arn']
print(role_arn)

arn:aws:iam::864107284170:role/lambda-launch-pipeline


In [17]:
# Attach logging (managed policy). It has 3 permissions : (logs:CreateLogGroup, logs:CreateLogStream, logs:PutLogEvents)
iam.attach_role_policy(
    RoleName='lambda-launch-pipeline',
    PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole',
)

{'ResponseMetadata': {'RequestId': '44673820-d953-4ae9-ae30-3710e393e89d',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Fri, 02 Oct 2026 16:14:04 GMT',
   'x-amzn-requestid': '44673820-d953-4ae9-ae30-3710e393e89d',
   'content-type': 'text/xml',
   'content-length': '212'},
  'RetryAttempts': 0}}

In [18]:
# Add the pipeline permission (inline policy)
iam.put_role_policy(
    RoleName='lambda-launch-pipeline',
    PolicyName='StartPipelineExecution',
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": "sagemaker:StartPipelineExecution",
            "Resource": "arn:aws:sagemaker:us-east-1:864107284170:pipeline/l3e3-pipeline"
        }]
    }),
)

{'ResponseMetadata': {'RequestId': 'be4d1c94-94fc-4b18-9808-34f5d0782c6f',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Fri, 02 Oct 2026 16:15:36 GMT',
   'x-amzn-requestid': 'be4d1c94-94fc-4b18-9808-34f5d0782c6f',
   'content-type': 'text/xml',
   'content-length': '206'},
  'RetryAttempts': 0}}

In [19]:
# Create the function

lam = boto3.client('lambda')

with open('lambda.zip', 'rb') as f:
    b_code = f.read()

lam.create_function(
    FunctionName='launchPipeline-eh',
    Runtime='python3.12',
    Handler='lambda_function.lambda_handler',
    Code={'ZipFile': b_code},
    Timeout=30,
    MemorySize=128,
    Publish=True,
    PackageType='Zip',
    Role=role_arn,
)

{'ResponseMetadata': {'RequestId': '2a291427-6958-4555-81c7-c29c8567ac47',
  'HTTPStatusCode': 201,
  'HTTPHeaders': {'date': 'Fri, 02 Oct 2026 16:19:59 GMT',
   'content-type': 'application/json',
   'content-length': '1406',
   'connection': 'keep-alive',
   'x-amzn-requestid': '2a291427-6958-4555-81c7-c29c8567ac47'},
  'RetryAttempts': 0},
 'FunctionName': 'launchPipeline-eh',
 'FunctionArn': 'arn:aws:lambda:us-east-1:864107284170:function:launchPipeline-eh',
 'Runtime': 'python3.12',
 'Role': 'arn:aws:iam::864107284170:role/lambda-launch-pipeline',
 'Handler': 'lambda_function.lambda_handler',
 'CodeSize': 525,
 'Description': '',
 'Timeout': 30,
 'MemorySize': 128,
 'LastModified': '2026-10-02T16:19:59.116+0000',
 'CodeSha256': 'XFK3B5zhEuAowD7SE+EUT8TiDl+0UyQ7WOXd18ca66Y=',
 'Version': '1',
 'TracingConfig': {'Mode': 'PassThrough'},
 'RevisionId': 'c789c764-3381-4c71-a79d-3f2d36188a7e',
 'State': 'Pending',
 'StateReason': 'The function is being created.',
 'StateReasonCode': 'Cr

In [20]:
# invoke the lambda function
resp = lam.invoke(FunctionName='launchPipeline-eh', Payload=b'{}')
print(resp.get('FunctionError'))
print(resp['Payload'].read())

None
b'{"statusCode": 200, "body": "{\\"message\\": \\"Pipeline execution started\\", \\"executionArn\\": \\"arn:aws:sagemaker:us-east-1:864107284170:pipeline/l3e3-pipeline/execution/rk5fnfach8ov\\"}"}'


## Exercise: Launch the Lambda Function


Launch the lambda function and confirm the step function is created successfully.

## Conceptual Exercise: What are next steps? 

Right now, the Step Function that we made in the prior exercise has a hard-coded location of the dataset we input, as well as all of the locations of the intermediary steps. What are ways that you could modify the Step Function to make it more generalizable? If you could input an S3 location, how could you integrate it with Lambda so that it could asychronously be called? 